In [1]:
print("Let's Start")

Let's Start


In [2]:
import pandas as pd

df = pd.read_excel(r"C:\Users\Abhishek sharma\Artificial Intelligence\Datasets\Covid_dataset_curated.xlsx")
df.head()

,Age,Gender,Travel History,Temp,SPO2,Contact to NCOVID Patient,Respiratory Support,Respiratory rate(breaths per minute),BMI,O2 supplementation required,...,breathlessness,cough,fever,headache,sore throat,asymptomatic,cold,malaise,myalgia,severity
0,53,1,0,96.8,99,1,1,20,22.5,1,...,1,1,1,0,0,0,0,0,0,0
1,26,0,0,98.7,98,1,0,16,25.7,0,...,0,0,0,0,0,1,0,0,0,0
2,28,1,0,98.4,98,1,0,16,22.2,0,...,0,0,0,0,0,1,0,0,0,0
3,73,1,0,98.0,98,1,1,26,21.5,1,...,1,1,1,0,0,0,0,0,0,1
4,49,1,0,101.0,98,1,0,20,27.4,0,...,0,0,1,0,0,0,0,0,0,0


In [3]:
x = df.iloc[:,:-1]
y = df.iloc[:,-1]


In [23]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import (
    train_test_split, cross_validate, learning_curve, validation_curve
)
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    classification_report, log_loss, mean_squared_error, r2_score , mean_absolute_error
)

from sklearn.calibration import calibration_curve

class BaseAnalyser:
    """Base class containing shared utilities for analysis."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def _create_dir(self, directory):
        if not os.path.exists(directory):
            os.makedirs(directory)

    def _ensure_dir(self, directory):
        if not os.path.exists(directory):
            os.makedirs(directory)

class ClassificationAnalyser(BaseAnalyser):
    """
    A toolset for evaluating and comparing classification models.
    This class provides methods to calculate cross-validation scores, 
    generate ROC curves, learning curves, and calibration plots using Plotly.
    """

    def cross_validator(self, model, cv=10):
        """
        Perform k-fold cross-validation with multiple classification metrics.

        Args:
            model: The scikit-learn compatible classifier instance.
            cv (int): Number of cross-validation folds. Defaults to 10.

        Returns:
            pd.DataFrame: Mean scores for accuracy, precision, recall, F1, and ROC-AUC.
        """
        scoring_metrics = {
            'acc': 'accuracy',
            'bal_acc': 'balanced_accuracy',
            'prec_macro': 'precision_macro',
            'rec_macro': 'recall_macro',
            'f1_macro': 'f1_macro',
            'f1_weighted': 'f1_weighted',
            'roc_auc': 'roc_auc'
        }
        scores = cross_validate(
            model, self.x, self.y, cv=cv,
            scoring=scoring_metrics, n_jobs=-1, error_score='raise'
        )
        return pd.DataFrame(pd.DataFrame(scores).mean(), columns=['Mean Score'])

    def score_comparison(self, models, test_size=0.3, random_state=42, filename="score_data.csv", save_file=False):
        """
        Compare multiple classifiers on various metrics and save to CSV.

        Args:
            models (list): List of classifier instances.
            test_size (float): Proportion of test data.
            random_state (int): Seed for reproducibility.
            filename (str): Path to save the CSV.
            save_file (bool): Whether to save the result.

        Returns:
            pd.DataFrame: Comparison table of all models.
        """
        rows = []
        x_train, x_test, y_train, y_test = train_test_split(
            self.x, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )

        for model in models:
            model.fit(x_train, y_train)
            y_pred = model.predict(x_test)
            cv_results = self.cross_validator(model)
            row = {
                'Model Names': type(model).__name__,
                'training' : model.score(x_train , y_train),
                'testing' : model.score(x_test , y_test),
                'Accuracy': cv_results.loc['test_acc'][0],
                'F1_Macro': cv_results.loc['test_f1_macro'][0],
                'ROC_AUC': cv_results.loc['test_roc_auc'][0],
                'Log_Loss': log_loss(y_test, model.predict_proba(x_test))
            }
            rows.append(row)
        df = pd.DataFrame(rows).set_index('Model Names')
        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                df.to_csv(file_path)
                print(f"✅ Success: Report saved at {file_path}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
        return df


    def classification_report_comparison(self, models, filename="classification_report.csv" , save_file=False):
        """
        Analyse and save Classification Report comparison.

        Args:
            models (list): List of fitted or unfitted classifiers.
            save_file (bool): Save the plot as an image.
            filename (str): Path for the image file.

        Returns:
            A .csv file having a comparison of classification report of different models
        """
        X_train, X_test, y_train, y_test = train_test_split(
            self.x, self.y, test_size=0.3,
            random_state=42, stratify=self.y
        )
        all_model_data = []

        for model in models:
            model_name = type(model).__name__
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            report = classification_report(y_test, y_pred, output_dict=True)
            model_row = {
                'Model Names': model_name,
                'Accuracy': report['accuracy'],
                'Precision (Macro)': report['macro avg']['precision'],
                'Recall (Macro)': report['macro avg']['recall'],
                'F1-Score (Macro)': report['macro avg']['f1-score'],
                'F1-Score (Weighted)': report['weighted avg']['f1-score']
            }

            all_model_data.append(model_row)
            print(f"✅ Evaluated: {model_name}")
        comparison_df = pd.DataFrame(all_model_data).set_index('Model Names')
        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                comparison_df.to_csv(file_path)
                print(f"✅ Success: Report saved at {file_path}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
            except Exception as e:
                print(f"{e}")
        return comparison_df

    def auc_curve_saver(self, models, test_size = 0.3 , random_state=42 , save_file = False , filename="roc_auc_model_comparisons.png"):
        """
        Plot and optionally save ROC-AUC curve comparison.

        Args:
            models (list): List of fitted or unfitted classifiers.
            save_file (bool): Save the plot as an image.
            filename (str): Path for the image file.
        """
        X_train , X_test , y_train , y_test = train_test_split(self.x , self.y , test_size=test_size , random_state=random_state , stratify=self.y)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines',
            name='Random (AUC = 0.5)',
            line=dict(dash='dash', color='grey')
        ))
        for model in models:
            model_name = type(model).__name__
            model.fit(X_train, y_train)

            y_proba = model.predict_proba(X_test)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            auc_score = roc_auc_score(y_test, y_proba)

            fig.add_trace(go.Scatter(
                x=fpr, y=tpr,
                mode='lines',
                name=f'{model_name} (AUC: {auc_score:.2f})'
            ))

        fig.update_layout(
            title='ROC Curve Comparison',
            xaxis_title='False Positive Rate',
            yaxis_title='True Positive Rate',
            template='plotly_white'
        )
        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                fig.write_image(file_path)
                print(f"✅ Success: Graph saved at {file_path}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
        fig.show()

    def caliberation_curve(self, models, test_size = 0.3 , random_state=42 , save_file = False, filename="caliberation_model_comparisons.png"):
        """
        Plot and optionally save Caliberation curve comparison.

        Args:
            models (list): List of fitted or unfitted classifiers.
            save_file (bool): Save the plot as an image.
            filename (str): Path for the image file.
        """
        X_train , X_test , y_train , y_test = train_test_split(self.x , self.y , test_size=test_size , random_state=random_state , stratify=self.y)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines',
            name='Random (AUC = 0.5)',
            line=dict(dash='dash', color='grey')
        ))
        for model in models:
            model_name = type(model).__name__
            model.fit(X_train, y_train)

            y_proba = model.predict_proba(X_test)[:, 1]
            prob_true , prob_pred = calibration_curve(y_test , y_proba)
            fig.add_trace(go.Scatter(
                x=prob_pred, y=prob_true,
                mode='lines',
                name=f'{model_name})'
            ))

            fig.update_layout(
            title='Caliberation Curve Comparison',
            xaxis_title='Predicted Probability',
            yaxis_title='True Probability',
            template='plotly_white'
            )

        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                fig.write_image(file_path)
                print(f"✅ Success: Graph saved at {file_path}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
            except Exception as e:
                print(f"{e}")
        fig.show()


    def learning_curve_saver(self, models, test_size=0.3, random_state=42,cv = 5 , save_file=False, filename="learning_curve.png", train_set=True , scoring='accuracy'):
        """
        Plot and optionally save Learning curve comparison.
        """
        fig = go.Figure()
        train_sizes = np.linspace(0.1, 1.0, 5)

        # FIX: Correct sequence is X_train, X_test, y_train, y_test
        x_train, x_test, y_train, y_test = train_test_split(
            self.x, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )

        for model in models:
            model_name = type(model).__name__
            # Choose which data to use for learning curve cross-validation
            data_x, data_y = (x_train, y_train) if train_set else (x_test, y_test)
            train_sizes_abs, train_scores, test_scores = learning_curve(
                model, data_x, data_y,
                train_sizes=train_sizes,
                cv=cv,
                scoring=scoring,
                n_jobs=-1
            )
            # 1. Dono scores ka mean nikalna zaroori hai
            train_scores_mean = np.mean(train_scores, axis=1)
            test_scores_mean = np.mean(test_scores, axis=1)

            # 2. Training Line (How well the model knows the training data)
            fig.add_trace(go.Scatter(
                x=train_sizes_abs,
                y=train_scores_mean,
                mode='lines+markers',
                name=f'{model_name} (Train)',
                line=dict(dash='dash') # Training line ko dotted rakhte hain contrast ke liye
            ))

            # 3. Validation Line (How well the model generalizes)
            fig.add_trace(go.Scatter(
                x=train_sizes_abs,
                y=test_scores_mean,
                mode='lines+markers',
                name=f'{model_name} (Val)'
            ))

        fig.update_layout(
            title=f"Learning Curves Comparison ({'Training' if train_set else 'Test'} Set)",
            xaxis_title='Number of Samples',
            yaxis_title='Accuracy Score',
            template='plotly_white',
            hovermode="x",
            width=900,
            height=600
        )

        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                fig.write_image(file_path)
                print(f"✅ Success: Learning Curve saved at reports/{file_path}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
            except Exception as e:
                print(f"{e}")
        fig.show()

    def validation_curve_plotter(self, model, param_name, param_range, cv=5, save_file=False, filename="val_curve.png"):
        """
        Plots the validation curve for a specific hyperparameter to analyze Bias-Variance tradeoff.

        Args:
            model: The classifier instance (e.g., RandomForestClassifier()).
            param_name (str): Name of the hyperparameter to vary (e.g., 'max_depth').
            param_range (list or np.array): The values of the parameter to test.
            cv (int): Number of cross-validation folds. Defaults to 5.
            save_file (bool): Whether to save the plot as an image.
            filename (str): Name of the file to save.
        """
        print(f"📊 Calculating Validation Curve for {param_name}...")
        # Validation curve calculate karna
        train_scores, test_scores = validation_curve(
            model, self.x, self.y, 
            param_name=param_name, 
            param_range=param_range,
            cv=cv, 
            scoring="accuracy", 
            n_jobs=-1
        )

        # Mean aur Standard Deviation nikalna
        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        test_mean = np.mean(test_scores, axis=1)
        test_std = np.std(test_scores, axis=1)

        fig = go.Figure()

        # Training Score Trace
        fig.add_trace(go.Scatter(
            x=param_range, y=train_mean,
            mode='lines+markers',
            name='Training Score',
            line=dict(color='blue')
        ))

        # Cross-Validation Score Trace
        fig.add_trace(go.Scatter(
            x=param_range, y=test_mean,
            mode='lines+markers',
            name='Cross-Validation Score',
            line=dict(color='green')
        ))

        fig.update_layout(
            title=f'Validation Curve for {type(model).__name__} ({param_name})',
            xaxis_title=f'Parameter: {param_name}',
            yaxis_title='Accuracy Score',
            template='plotly_white',
            hovermode='x unified'
        )

        if save_file:
            self._ensure_dir('reports')
            try:
                file_path = f"reports/{filename}"
                fig.write_image(file_path)
                print(f"✅ Validation Curve saved at: results/{filename}")
            except PermissionError:
                print(f"❌ Error: Please close '{filename}' if it is open in Excel and try again.")
            except Exception as e:
                print(f"{e}")


        fig.show()

class RegressionAnalyser(BaseAnalyser):
    """
    Dedicated toolset for Regression tasks.
    Includes Error analysis, Residual plots, and Prediction vs Actual comparisons.
    """

    def regression_metrics(self, model, test_size=0.3, random_state=42):
        """
        Calculate standard regression metrics (MSE, RMSE, MAE, R2).

        Args:
            model: Scikit-learn regressor.
            test_size (float): Test split ratio.
            random_state (int): Seed value.

        Returns:
            dict: Dictionary of calculated metrics.
        """
        x_train, x_test, y_train, y_test = train_test_split(self.x, self.y, test_size=test_size, random_state=random_state)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        return {
            'MSE': mean_squared_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        }

    def plot_regression_results(self, model, save_file=False, filename="results/regression_analysis.png"):
        """
        Generate two key regression plots: Prediction vs Actual and Residual Plot.

        Args:
            model: Regressor instance.
            save_file (bool): Save image if True.
            filename (str): Output path.
        """
        x_train, x_test, y_train, y_test = train_test_split(self.x, self.y, test_size=0.3)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        residuals = y_test - y_pred

        fig = make_subplots(rows=1, cols=2, subplot_titles=("Actual vs Predicted", "Residual Plot"))

        # Plot 1: Actual vs Predicted
        fig.add_trace(go.Scatter(x=y_test, y=y_pred, mode='markers', name='Predictions'), row=1, col=1)
        fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()], name='Perfect Fit', line=dict(color='red')), row=1, col=1)

        # Plot 2: Residuals
        fig.add_trace(go.Scatter(x=y_pred, y=residuals, mode='markers', name='Residuals'), row=1, col=2)
        fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=2)

        fig.update_layout(height=500, title_text=f"Analysis: {type(model).__name__}", template="plotly_white")
        if save_file:
            self._ensure_dir(filename)
            fig.write_image(filename)
        fig.show()



    def validation_curve_plotter(self, model, param_name, param_range, cv=5, scoring="r2", save_file=False, filename="results/val_curve.png"):
        """
        Plot Validation Curve to analyze bias-variance tradeoff for a parameter.

        Args:
            model: Regressor or Classifier.
            param_name (str): Parameter to tune.
            param_range (list): Values to test.
            scoring (str): Metric to use (e.g., 'r2' or 'neg_mean_squared_error').
        """
        train_scores, test_scores = validation_curve(
            model, self.x, self.y, param_name=param_name, param_range=param_range,
            cv=cv, scoring=scoring, n_jobs=-1
        )

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=param_range, y=np.mean(train_scores, axis=1), name="Train Score"))
        fig.add_trace(go.Scatter(x=param_range, y=np.mean(test_scores, axis=1), name="Cross-Val Score"))

        fig.update_layout(title=f"Validation Curve ({param_name})", xaxis_title=param_name, yaxis_title=scoring)
        if save_file:
            self._ensure_dir(filename)
            fig.write_image(filename)
        fig.show()

In [ ]:
import pickle
scores = pickle.load(open(r"C:\Users\Abhishek sharma\Artificial Intelligence\Machine Learning\Projects\ml-automator\tests\best_scores.pkl" , 'rb'))

{'RandomForest': {'n_estimators': 352,
  'max_depth': 40,
  'min_samples_split': 9,
  'min_samples_leaf': 8,
  'max_features': None,
  'bootstrap': True},
 'XGBoost': {'n_estimators': 781,
  'learning_rate': 0.033523219289001406,
  'max_depth': 15,
  'subsample': 0.9962411422735371,
  'colsample_bytree': 0.4055366014266707,
  'gamma': 5.431507273155752e-05,
  'min_child_weight': 1},
 'AdaBoost': {'n_estimators': 406, 'learning_rate': 1.2275826710862345},
 'LightGBM': {'n_estimators': 493,
  'learning_rate': 0.007709391232564028,
  'num_leaves': 59,
  'max_depth': 17,
  'subsample': 0.44473038620786254,
  'colsample_bytree': 0.9921321619603104,
  'reg_alpha': 0.08916674715636537,
  'reg_lambda': 6.143857495033091e-07},
 'CatBoost': {'iterations': 624,
  'learning_rate': 0.20218499516556748,
  'depth': 9,
  'l2_leaf_reg': 0.6251373574521749,
  'random_strength': 2.5361081166471375e-07,
  'bagging_temperature': 0.15599452033620265}}

In [13]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

models = [RandomForestClassifier(**scores['RandomForest']) , AdaBoostClassifier(**scores['AdaBoost']) , lgb.LGBMClassifier(**scores['LightGBM']) , xgb.XGBClassifier(**scores['XGBoost']) , CatBoostClassifier(**scores['CatBoost'])]
ca = ClassificationAnalyser(x , y)

In [14]:
ca.validation_curve_plotter(models[0] , cv =10 , param_name= 'max_depth' , param_range=[1,2,3,4,5 , 6 ,7 ,8 ,9, 10] , save_file=False , filename='val_curve-before_finetuning.png')

📊 Calculating Validation Curve for max_depth...


In [24]:
ca.learning_curve_saver(models , save_file=True  , filename="learning_curve_after_finetuning.png")

✅ Success: Learning Curve saved at reports/reports/learning_curve_after_finetuning.png


In [16]:
ca.caliberation_curve(models , save_file=True , filename='caliberation_after_finetunig.png')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 15, number of negative: 90
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000242 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 534
[LightGBM] [Info] Number of data points in the train set: 105, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.142857 -> initscore=-1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [53]:
ca.auc_curve_saver(models , save_file=True , filename='roc_auc_before_tuning.png')

✅ Success: Graph saved at reports/roc_auc_before_tuning.png


In [17]:
ca.classification_report_comparison(models , save_file=True , filename='classification_report_after_finetuning.csv')

✅ Evaluated: RandomForestClassifier
✅ Evaluated: AdaBoostClassifier
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 15, number of negative: 90
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000638 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 534
[LightGBM] [Info] Number of data points in the train set: 105, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.142857 -> initscore=-1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,Accuracy,Precision (Macro),Recall (Macro),F1-Score (Macro),F1-Score (Weighted)
Model Names,,,,,
RandomForestClassifier,0.869565,0.933333,0.571429,0.589286,0.825311
AdaBoostClassifier,0.934783,0.964286,0.785714,0.845118,0.927097
LGBMClassifier,0.913043,0.953488,0.714286,0.775610,0.897773
XGBClassifier,0.891304,0.943182,0.642857,0.692102,0.864385
CatBoostClassifier,0.847826,0.681818,0.558608,0.568942,0.810139


In [21]:

ca.score_comparison(models , save_file=True , filename='score_data_after_finetuning.csv')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 15, number of negative: 90
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000189 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 534
[LightGBM] [Info] Number of data points in the train set: 105, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.142857 -> initscore=-1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

,Accuracy,F1_Macro,ROC_AUC,Log_Loss
Model Names,,,,
RandomForestClassifier,0.927500,0.866500,0.981624,0.252993
AdaBoostClassifier,0.967083,0.927111,0.989744,0.280211
LGBMClassifier,0.953750,0.903547,0.989423,0.247610
XGBClassifier,0.960417,0.911759,0.992094,0.252168
CatBoostClassifier,0.933750,0.858288,0.980769,0.865257


In [8]:
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score , log_loss
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from concurrent.futures import ThreadPoolExecutor
import warnings

# Clean terminal output
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings('ignore')


class ClassificationTuner:
    """
    Hyperparameter Tuning Engine for Classification tasks using Optuna.
    """

    def __init__(self,x , y ,  n_trials=50,  x_test = 0.2 ,  seed=42 ):
        self.n_trials = n_trials
        self.x = x
        self.y = y
        self.seed = seed
        self.x_test = x_test
        self.best_configs = {} # Final dictionary for all results

    # --- Objective Functions for Each Model ---

    def __splitter(self):
        x_train , x_test , y_train , y_test = train_test_split(self.x , self.y , test_size=self.x_test , random_state=self.seed , stratify=self.y)
        return x_train , x_test , y_train , y_test

    def __dt_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss']),
            'max_depth': trial.suggest_int('max_depth', 2, 32),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'random_state': self.seed
        }
        return self.__evaluate(DecisionTreeClassifier(**params), x_t, y_t, x_v, y_v)

    def __svm_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'C': trial.suggest_float('C', 1e-4, 100.0, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'probability': True, # Log-loss ke liye probability zaroori hai
            'random_state': self.seed
        }
        return self.__evaluate(SVC(**params), x_t, y_t, x_v, y_v)

    def __knn_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
            'n_jobs': -1
        }
        return self.__evaluate(KNeighborsClassifier(**params), x_t, y_t, x_v, y_v)

    def __ada_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 2.0, log=True),
            'random_state': self.seed
        }
        # AdaBoost boosting model hai par ye eval_set support nahi karta 
        # isliye is_boost=False rahega
        return self.__evaluate(AdaBoostClassifier(**params), x_t, y_t, x_v, y_v, is_boost=False)


    def __rf_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
            'max_depth': trial.suggest_int('max_depth', 2, 64),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'n_jobs': -1, 'random_state': self.seed
        }
        return self.__evaluate(RandomForestClassifier(**params), x_t, y_t, x_v, y_v)

    def __xgb_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'subsample': trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 10.0, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'random_state': self.seed, 'verbosity': 0
        }
        return self.__evaluate(xgb.XGBClassifier(**params), x_t, y_t, x_v, y_v, is_boost=True)

    def __lgb_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'subsample': trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': self.seed
        }
        return self.__evaluate(lgb.LGBMClassifier(**params), x_t, y_t, x_v, y_v, is_boost=True)

    def __cat_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'iterations': trial.suggest_int('iterations', 100, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'verbose': 0, 'random_state': self.seed
        }
        return self.__evaluate(CatBoostClassifier(**params), x_t, y_t, x_v, y_v, is_boost=True)

    def __evaluate(self, model, x_t, y_t, x_v, y_v, is_boost=False, metric='f1'):
        """
        Model ko train aur evaluate karne ka central point.
        Handled: LGBM verbose error and Multi-metric support.
        """
        import lightgbm as lgb

        if is_boost:
            # Check if model is LightGBM to use callbacks instead of verbose
            if isinstance(model, (lgb.LGBMClassifier)):
                model.fit(
                    x_t, y_t,
                    eval_set=[(x_v, y_v)],
                    callbacks=[lgb.log_evaluation(period=0)] # Modern way to silence logs
                )
            else:
                # XGBoost and CatBoost still support verbose=False in many versions
                model.fit(x_t, y_t, eval_set=[(x_v, y_v)], verbose=False)
        else:
            model.fit(x_t, y_t)

        # Metric Logic
        if metric == 'f1':
            preds = model.predict(x_v)
            # 'macro' use kar rahe hain taaki saari classes ko barabar weight mile
            return f1_score(y_v, preds, average='macro')
        elif metric == 'logloss':
            try:
                probs = model.predict_proba(x_v)
                return -log_loss(y_v, probs)
            except AttributeError:
                return -1.0
        else: # accuracy
            preds = model.predict(x_v)
            return accuracy_score(y_v, preds)

    # --- Core Tuning Logic ---
    def __tune_single_model(self, model_key , x_train , y_train , x_val , y_val):
        """
        Kisi ek model ko tune karne ke liye.
        model_key: 'rf', 'xgb', 'lgb', 'cat', 'ada', 'dt', 'svm', 'knn'
        """

        # 1. Available models ki dictionary
        objectives = {
            'rf': ('RandomForest', self.__rf_obj),
            'xgb': ('XGBoost', self.__xgb_obj),
            'lgb': ('LightGBM', self.__lgb_obj),
            'cat': ('CatBoost', self.__cat_obj),
            'ada': ('AdaBoost', self.__ada_obj),
            'dt': ('DecisionTree', self.__dt_obj),
            'svm': ('SVM', self.__svm_obj),
            'knn': ('KNN', self.__knn_obj)
        }

        # 2. Check karein ki user ne sahi key dali hai ya nahi
        if model_key not in objectives:
            available_keys = ", ".join([f"'{k}'" for k in objectives.keys()])
            # VS Code ya terminal mein ye error user ko guide karega
            raise ValueError(
                f"❌ Invalid model_key: '{model_key}'. "
                f"Please choose from the following valid options: {available_keys}"
            )

        # 3. Agar sahi hai toh tuning shuru karein
        name, obj_func = objectives[model_key]
        print(f"\n🚀 Starting Tuning for: {name}")
        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=self.seed))
        study.optimize(lambda t: obj_func(t, x_train, y_train, x_val, y_val), n_trials=self.n_trials)
        print(f"✅ {name} Tuning Complete. Best Accuracy: {study.best_value:.4f}")
        return {name: study.best_params}


    def tune(self, model_keys=['rf', 'xgb', 'lgb', 'cat', 'ada', 'dt', 'svm', 'knn']):
        """
        Runs parallel tuning for classification task

        Returns
            Dictionary of best scores of given models in model_keys
        """
        # 1. Valid keys ki list for suggestion
        all_valid_keys = ['rf', 'xgb', 'lgb', 'cat', 'ada', 'dt', 'svm', 'knn']

        # 2. Validation check
        for key in model_keys:
            if key not in all_valid_keys:
                raise ValueError(
                    f"❌ Invalid model key: '{key}'.\n"
                    f"Available options are: {all_valid_keys}"
                )

        # 3. Data split (aapka internal splitter method)
        x_train, x_val,y_train , y_val = self.__splitter()
        # Purane results clear karein taaki nayi tuning fresh ho
        self.best_configs = {}

        # 1. Total Cores check karein (e.g., 8 or 16)
        total_cores = os.cpu_count() or 4
        num_models = len(model_keys)

        # 2. Smart Allocation: Har model ko kitne cores milenge?
        # Agar 8 cores hain aur 4 models, toh har model ko 2 cores milenge.
        cores_per_model = max(1, total_cores // num_models)

        print(f"🖥️  System Detected: {total_cores} Cores")
        print(f"🚀 Allocating {cores_per_model} core(s) per model for {num_models} models.")

        print(f"🚀 Starting parallel tuning for: {model_keys}")

        # 4. Parallel execution using ThreadPoolExecutor
        with ThreadPoolExecutor() as executor:
            # Note: self._tune_single_model ko call kar rahe hain
            futures = [executor.submit(self.__tune_single_model, key, x_train, y_train, x_val, y_val) for key in model_keys]
            for f in futures:
                result = f.result()
                if result:
                    self.best_configs.update(result)
        print("\n✅ All specified models tuned successfully!")
        return self.best_configs



class RegressionTuner:
    """
    Hyperparameter Tuning Engine for Regression tasks.
    Optimizes for Mean Squared Error (MSE) using Optuna.
    """
    def __init__(self, x, y, n_trials=50, test_size=0.2, seed=42):
        self.x = x
        self.y = y
        self.n_trials = n_trials
        self.test_size = test_size
        self.seed = seed
        self.best_configs = {}

    def __splitter(self):
        # Note: No stratification for regression targets
        return train_test_split(self.x, self.y, test_size=self.test_size, random_state=self.seed)

    def __evaluate(self, model, x_t, y_t, x_v, y_v, is_boost=False):
        """Calculates MSE for Optuna to minimize."""
        if is_boost:
            model.fit(x_t, y_t, eval_set=[(x_v, y_v)], verbose=False)
        else:
            model.fit(x_t, y_t)
        preds = model.predict(x_v)
        return mean_squared_error(y_v, preds)

    # --- Hidden Objective Functions ---

    def __rf_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 800),
            'max_depth': trial.suggest_int('max_depth', 2, 32),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'random_state': self.seed
        }
        return self.__evaluate(RandomForestRegressor(**params), x_t, y_t, x_v, y_v)

    def __xgb_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'random_state': self.seed, 'verbosity': 0
        }
        return self.__evaluate(xgb.XGBRegressor(**params), x_t, y_t, x_v, y_v, is_boost=True)

    def __lgb_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 256),
            'random_state': self.seed, 'verbose': -1
        }
        model = lgb.LGBMRegressor(**params)
        model.fit(x_t, y_t, eval_set=[(x_v, y_v)], callbacks=[lgb.log_evaluation(period=0)])
        return mean_squared_error(y_v, model.predict(x_v))

    def __cat_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'iterations': trial.suggest_int('iterations', 100, 1000),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'verbose': 0, 'random_seed': self.seed
        }
        return self.__evaluate(CatBoostRegressor(**params), x_t, y_t, x_v, y_v, is_boost=True)

    def __svm_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'C': trial.suggest_float('C', 1e-3, 100, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 1.0), # Regression specific
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])
        }
        return self.__evaluate(SVR(**params), x_t, y_t, x_v, y_v)

    def __ada_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1.0, log=True),
            'loss': trial.suggest_categorical('loss', ['linear', 'square', 'exponential']),
            'random_state': self.seed
        }
        return self.__evaluate(AdaBoostRegressor(**params), x_t, y_t, x_v, y_v)

    def __dt_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'max_depth': trial.suggest_int('max_depth', 2, 32),
            'criterion': trial.suggest_categorical('criterion', ['squared_error', 'absolute_error', 'friedman_mse'])
        }
        return self.__evaluate(DecisionTreeRegressor(**params), x_t, y_t, x_v, y_v)

    def __knn_obj(self, trial, x_t, y_t, x_v, y_v):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance'])
        }
        return self.__evaluate(KNeighborsRegressor(**params), x_t, y_t, x_v, y_v)

    # --- Core Logic ---

    def __tune_single_model(self, model_key, x_t, y_t, x_v, y_v):
        objectives = {
            'rf': ('RandomForest', self.__rf_obj),
            'xgb': ('XGBoost', self.__xgb_obj),
            'lgb': ('LightGBM', self.__lgb_obj),
            'cat': ('CatBoost', self.__cat_obj),
            'ada': ('AdaBoost', self.__ada_obj),
            'dt': ('DecisionTree', self.__dt_obj),
            'svm': ('SVM', self.__svm_obj),
            'knn': ('KNN', self.__knn_obj)
        }
        name, obj_func = objectives[model_key]
        # Direction 'minimize' for Error
        study = optuna.create_study(direction='minimize')
        study.optimize(lambda t: obj_func(t, x_t, y_t, x_v, y_v), n_trials=self.n_trials)
        return {name: (study.best_params, study.best_value)}

    def tune(self, model_keys=['rf', 'xgb', 'lgb', 'cat', 'ada', 'dt', 'svm', 'knn']):
        """
        Runs parallel tuning for regression models.

        Returns
            Dictionary of best scores of given models in model_keys
        """
        x_train, x_test, y_train, y_test = self.__splitter()
        self.best_configs = {}

        print(f"🚀 Starting Parallel Regression Tuning for: {model_keys}")

        with ThreadPoolExecutor() as executor:
            futures = [
                executor.submit(self.__tune_single_model, key, x_train, y_train, x_test, y_test) 
                for key in model_keys
            ]
            for f in futures:
                self.best_configs.update(f.result())

        print(f"✅ Tuning complete!")
        return self.best_configs

In [7]:
if __name__ == "__main__":
    class_tuner = ClassificationTuner(x , y)
    bc = class_tuner.tune(['rf' , 'xgb' , 'ada' , 'lgb' , "cat"])

🖥️  System Detected: 12 Cores
🚀 Allocating 2 core(s) per model for 5 models.
🚀 Starting parallel tuning for: ['rf', 'xgb', 'ada', 'lgb', 'cat']

🚀 Starting Tuning for: RandomForest

🚀 Starting Tuning for: XGBoost

🚀 Starting Tuning for: AdaBoost

🚀 Starting Tuning for: LightGBM

🚀 Starting Tuning for: CatBoost
✅ RandomForest Tuning Complete. Best Accuracy: 0.7584
✅ LightGBM Tuning Complete. Best Accuracy: 0.7584
✅ XGBoost Tuning Complete. Best Accuracy: 0.7584


TypeError: ClassificationTuner.__ada_obj() takes 6 positional arguments but 7 were given

In [34]:
bc

{'RandomForest': {'n_estimators': 352,
  'max_depth': 40,
  'min_samples_split': 9,
  'min_samples_leaf': 8,
  'max_features': None,
  'bootstrap': True},
 'XGBoost': {'n_estimators': 781,
  'learning_rate': 0.033523219289001406,
  'max_depth': 15,
  'subsample': 0.9962411422735371,
  'colsample_bytree': 0.4055366014266707,
  'gamma': 5.431507273155752e-05,
  'min_child_weight': 1},
 'AdaBoost': {'n_estimators': 406, 'learning_rate': 1.2275826710862345},
 'LightGBM': {'n_estimators': 493,
  'learning_rate': 0.007709391232564028,
  'num_leaves': 59,
  'max_depth': 17,
  'subsample': 0.44473038620786254,
  'colsample_bytree': 0.9921321619603104,
  'reg_alpha': 0.08916674715636537,
  'reg_lambda': 6.143857495033091e-07},
 'CatBoost': {'iterations': 624,
  'learning_rate': 0.20218499516556748,
  'depth': 9,
  'l2_leaf_reg': 0.6251373574521749,
  'random_strength': 2.5361081166471375e-07,
  'bagging_temperature': 0.15599452033620265}}

In [16]:
import pickle

pickle.dump(bc , open('best_scores.pkl' , 'wb'))

In [19]:
scores = pickle.load(open(r"C:\Users\Abhishek sharma\Artificial Intelligence\Machine Learning\Projects\ml-automator\tests\best_scores.pkl" , 'rb'))
scores

{'RandomForest': {'n_estimators': 352,
  'max_depth': 40,
  'min_samples_split': 9,
  'min_samples_leaf': 8,
  'max_features': None,
  'bootstrap': True},
 'XGBoost': {'n_estimators': 781,
  'learning_rate': 0.033523219289001406,
  'max_depth': 15,
  'subsample': 0.9962411422735371,
  'colsample_bytree': 0.4055366014266707,
  'gamma': 5.431507273155752e-05,
  'min_child_weight': 1},
 'AdaBoost': {'n_estimators': 406, 'learning_rate': 1.2275826710862345},
 'LightGBM': {'n_estimators': 493,
  'learning_rate': 0.007709391232564028,
  'num_leaves': 59,
  'max_depth': 17,
  'subsample': 0.44473038620786254,
  'colsample_bytree': 0.9921321619603104,
  'reg_alpha': 0.08916674715636537,
  'reg_lambda': 6.143857495033091e-07},
 'CatBoost': {'iterations': 624,
  'learning_rate': 0.20218499516556748,
  'depth': 9,
  'l2_leaf_reg': 0.6251373574521749,
  'random_strength': 2.5361081166471375e-07,
  'bagging_temperature': 0.15599452033620265}}